# Modele nieparametryczne – zbiór danych Heart Failure

Analiza czasu przeżycia metodami nieparametrycznymi: Kaplan-Meier, tablice trwania życia oraz wygładzanie kernelowe funkcji hazardu.

- **Zmienna czasu**: `time` (dni)
- **Zdarzenie**: `DEATH_EVENT = 1` (zgon)
- **N = 299** obserwacji; 96 zgonów (32,1 %)

In [ ]:
#!pip install lifelines pandas matplotlib numpy seaborn openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, NelsonAalenFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import logrank_test, multivariate_logrank_test
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set(font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('heart_failure_clinical_records_dataset.csv')
print('Ksztalt danych:', df.shape)
display(df[['time', 'DEATH_EVENT']].describe().round(2))
event_rate = df['DEATH_EVENT'].mean()
print(f'Czestotliwosc zdarzen: {event_rate:.3f} '
      f"({df['DEATH_EVENT'].sum()} z {len(df)})")


## KOD 1.1 – Podstawowe statystyki i korelacje z czasem przeżycia

In [ ]:
selected_cols = ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
                 'ejection_fraction', 'high_blood_pressure', 'platelets',
                 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time']

correlations = df[selected_cols].corr()['time'].sort_values(ascending=False).drop('time')
corr_df = correlations.reset_index()
corr_df.columns = ['Zmienna', 'Korelacja z czasem przezycia']

print('Korelacje zmiennych z czasem przezycia:')
display(corr_df.style
        .background_gradient(cmap='coolwarm',
                             subset=['Korelacja z czasem przezycia'])
        .format({'Korelacja z czasem przezycia': '{:.4f}'}))

plt.figure(figsize=(8, 6))
sns.barplot(data=corr_df, y='Zmienna',
            x='Korelacja z czasem przezycia',
            palette='coolwarm_r', orient='h')
plt.title('Korelacje zmiennych z czasem przezycia (Heart Failure)')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('hf_correlations_with_time.png', dpi=150)
plt.show()


## KOD 1.2 – Krzywa Kaplana-Meiera z przedziałami ufności i tabelą osób narażonych

Estymator K-M: $\hat{S}(t) = \prod_{t_i \leq t} \left(1 - \frac{d_i}{n_i}\right)$, gdzie $d_i$ – liczba zdarzeń, $n_i$ – liczba narażonych w chwili $t_i$.

In [ ]:
plt.figure(figsize=(12, 8))
ax = plt.subplot(111)

kmf = KaplanMeierFitter()
kmf.fit(df['time'], df['DEATH_EVENT'], label='Estymator K-M', alpha=0.05)
kmf.plot(ax=ax, ci_show=True, ci_alpha=0.3)
add_at_risk_counts(kmf, ax=ax)

plt.title('Krzywa przezycia Kaplana-Meiera z 95% przedzialem ufnosci\n'
          'Heart Failure (N=299, zdarzenia=96)')
plt.xlabel('Czas (dni)')
plt.ylabel('Prawdopodobienstwo przezycia')
plt.grid(True)
plt.tight_layout()
plt.savefig('hf_km_overall.png', dpi=150)
plt.show()

print(f'Mediana czasu przezycia (K-M): {kmf.median_survival_time_:.1f} dni')


## KOD 1.3 – Tabela przeżycia Kaplana-Meiera

In [ ]:
kmf_table = pd.DataFrame(kmf.survival_function_)
kmf_table.columns = ['Przezycie']
kmf_table['Dolny PU 95%'] = kmf.confidence_interval_.iloc[:, 0]
kmf_table['Gorny PU 95%'] = kmf.confidence_interval_.iloc[:, 1]
log_s = -np.log(np.clip(kmf.survival_function_.values.flatten(), 1e-10, 1))
kmf_table['Skum. hazard H(t)'] = log_s
kmf_table.loc[kmf_table.index[0], 'Skum. hazard H(t)'] = 0
kmf_table['Zdarzenia'] = kmf.event_table['observed']
kmf_table['Cenzurowane'] = kmf.event_table['censored']
kmf_table['Narazeni na ryzyko'] = kmf.event_table['at_risk']

print('Tabela przezycia K-M (pierwsze 15 wierszy):')
display(kmf_table.head(15))
print('\nTabela przezycia K-M (ostatnie 10 wierszy):')
display(kmf_table.tail(10))

kmf_table.to_excel('hf_km_table.xlsx')
print('Tabela zapisana do hf_km_table.xlsx')


## KOD 1.4 – Porównanie krzywych K-M wg zmiennych binarnych

Porównujemy krzywe przeżycia dla wszystkich zmiennych binarnych: `sex`, `anaemia`, `diabetes`, `high_blood_pressure`, `smoking`. Weryfikacja testu log-rank: $H_0$: krzywe przeżycia są identyczne.

In [ ]:
binary_vars = {
    'sex':                 {0: 'Kobieta', 1: 'Mezczyzna'},
    'anaemia':             {0: 'Brak anemii', 1: 'Anemia'},
    'diabetes':            {0: 'Brak cukrzycy', 1: 'Cukrzyca'},
    'high_blood_pressure': {0: 'Brak NT', 1: 'Nadcisnienie'},
    'smoking':             {0: 'Niepalacy', 1: 'Palacy'}
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (var, labels_map) in enumerate(binary_vars.items()):
    ax = axes[idx]
    for val in sorted(df[var].unique()):
        mask = df[var] == val
        kmf_g = KaplanMeierFitter()
        kmf_g.fit(df['time'][mask], df['DEATH_EVENT'][mask],
                  label=labels_map.get(val, str(val)))
        kmf_g.plot(ax=ax, ci_show=True)
    lr = multivariate_logrank_test(df['time'], df[var], df['DEATH_EVENT'])
    ax.set_title(f'{var}\nlog-rank p = {lr.p_value:.4f}')
    ax.set_xlabel('Czas (dni)')
    ax.set_ylabel('S(t)')
    ax.grid(True)

axes[-1].set_visible(False)
plt.suptitle('Porownanie krzywych K-M wg zmiennych binarnych (Heart Failure)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('hf_km_binary_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Szczegolowe wyniki testow log-rank
print('=== Testy log-rank i Wilcoxona dla zmiennych binarnych ===')
for var, labels_map in binary_vars.items():
    lr = multivariate_logrank_test(df['time'], df[var], df['DEATH_EVENT'])
    wx = multivariate_logrank_test(df['time'], df[var], df['DEATH_EVENT'],
                                   weightings='wilcoxon')
    print(f'\n{var}:')
    print(f'  Log-rank:  p = {lr.p_value:.4f} '
          f'({"istotny" if lr.p_value < 0.05 else "nieistotny"})')
    print(f'  Wilcoxon:  p = {wx.p_value:.4f} '
          f'({"istotny" if wx.p_value < 0.05 else "nieistotny"})')


## KOD 1.5 – Porównanie K-M wg kategoryzowanej frakcji wyrzutowej

Frakcja wyrzutowa (EF) jest kluczowym klinicznym predyktorem. Kategoryzujemy ją na kwartyle i porównujemy krzywe przeżycia.

In [ ]:
df['ef_cat'] = pd.qcut(df['ejection_fraction'], q=4,
                        labels=['Q1 (niska EF)', 'Q2', 'Q3', 'Q4 (wysoka EF)'])

plt.figure(figsize=(12, 8))
ax = plt.subplot(111)
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

kmf_list = []
for i, cat in enumerate(df['ef_cat'].cat.categories):
    mask = df['ef_cat'] == cat
    kmf_ef = KaplanMeierFitter()
    kmf_ef.fit(df['time'][mask], df['DEATH_EVENT'][mask], label=str(cat))
    kmf_ef.plot(ax=ax, ci_show=True, color=colors[i])
    kmf_list.append(kmf_ef)

lr_ef = multivariate_logrank_test(df['time'], df['ef_cat'], df['DEATH_EVENT'])

add_at_risk_counts(*kmf_list, ax=ax)
plt.title(f'Krzywa K-M wg frakcji wyrzutowej (kwartyle)\n'
          f'log-rank p = {lr_ef.p_value:.4f}')
plt.xlabel('Czas (dni)')
plt.ylabel('Prawdopodobienstwo przezycia')
plt.grid(True)
plt.tight_layout()
plt.savefig('hf_km_ef_quartiles.png', dpi=150)
plt.show()


## KOD 2 – Tablica trwania życia (Life Table)

Tablice trwania życia są starszą metodą nieparametryczną, grupującą obserwacje w przedziały czasu i obliczającą funkcję przeżycia oraz hazard dla każdego przedziału.

In [ ]:
def create_life_table(durations, events, intervals=None):
    if intervals is None:
        max_dur = max(durations)
        intervals = np.linspace(0, max_dur, 11)[1:]

    lt = pd.DataFrame({
        'Poczatek': [0] + list(np.ceil(intervals)[:-1]),
        'Koniec':    list(np.floor(intervals)),
        'Srodek (t_jm)':  np.zeros(len(intervals)),
        'Szerokosc (b_j)': np.zeros(len(intervals)),
        'Narazeni (n_j)': np.zeros(len(intervals)),
        'Zdarzenia (d_j)': np.zeros(len(intervals)),
        'Cenzurowane (m_j)': np.zeros(len(intervals)),
        'Hazard (h_j)':   np.zeros(len(intervals)),
        'P(zdarzenia) q_j': np.zeros(len(intervals)),
        'P(przezycia) p_j': np.zeros(len(intervals)),
        'Przezycie S(t)':   np.zeros(len(intervals)),
    })

    for i, (start, end) in enumerate(zip(lt['Poczatek'], lt['Koniec'])):
        in_interval = [(d > start) & (d <= end) for d in durations]
        at_risk     = sum([d > start for d in durations])
        events_i    = sum([e and (start < d <= end)
                           for d, e in zip(durations, events)])
        censored_i  = sum([(not e) and (start < d <= end)
                           for d, e in zip(durations, events)])
        width       = end - start
        mid         = (start + end) / 2
        eff_at_risk = at_risk - censored_i / 2
        eff_at_risk = max(eff_at_risk, 0.001)

        lt.at[i, 'Srodek (t_jm)']   = mid
        lt.at[i, 'Szerokosc (b_j)'] = width
        lt.at[i, 'Narazeni (n_j)']  = at_risk
        lt.at[i, 'Zdarzenia (d_j)'] = events_i
        lt.at[i, 'Cenzurowane (m_j)'] = censored_i
        lt.at[i, 'P(zdarzenia) q_j'] = events_i / eff_at_risk
        lt.at[i, 'P(przezycia) p_j'] = 1 - events_i / eff_at_risk
        lt.at[i, 'Hazard (h_j)']    = events_i / (eff_at_risk * width)

    S = 1.0
    for i in range(len(lt)):
        S *= lt.at[i, 'P(przezycia) p_j']
        lt.at[i, 'Przezycie S(t)'] = S

    return lt

# Generowanie tablic zycia
custom_intervals = [30, 60, 90, 120, 150, 180, 210, 240, 270, 285]
lt = create_life_table(df['time'].tolist(), df['DEATH_EVENT'].tolist(),
                        intervals=custom_intervals)

print('Tablica trwania zycia (Heart Failure):')
display(lt.round(4))
lt.to_excel('hf_life_table.xlsx', index=False)
print('Tablica zapisana do hf_life_table.xlsx')


In [ ]:
# Wykres funkcji przezycia i hazardu z tablicy zycia
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(lt['Koniec'], lt['Przezycie S(t)'], '-o',
             color='#1f77b4', linewidth=2, markersize=6)
axes[0].set_title('Funkcja przezycia z tablicy zycia (Heart Failure)')
axes[0].set_xlabel('Czas (dni)')
axes[0].set_ylabel('Prawdopodobienstwo przezycia S(t)')
axes[0].set_ylim(0, 1)
axes[0].grid(True)

axes[1].plot(lt['Koniec'], lt['Hazard (h_j)'], '-o',
             color='#d62728', linewidth=2, markersize=6)
axes[1].set_title('Funkcja hazardu z tablicy zycia (Heart Failure)')
axes[1].set_xlabel('Czas (dni)')
axes[1].set_ylabel('Hazard h(t)')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('hf_life_table_plots.png', dpi=150)
plt.show()


## KOD 3 – Logarytm funkcji przeżycia i diagnostyka rozkładu

Wykresy $-\log(\hat{S}(t))$ oraz $\log(-\log(\hat{S}(t)))$ vs $\log(t)$ służą diagnostyce zgodności z rozkładami parametrycznymi:
- Liniowość $-\log(S(t))$ vs $t$ → rozkład wykładniczy
- Liniowość $\log(-\log(S(t)))$ vs $\log(t)$ → rozkład Weibulla
- Proporcjonalność hazardów: równoległe krzywe $\log(-\log(S(t)))$ dla grup

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# -log(S(t))
log_s = -np.log(np.clip(kmf.survival_function_.values.flatten(), 1e-10, 1))
axes[0].plot(kmf.survival_function_.index, log_s, 'o-', color='#1f77b4')
axes[0].set_title('-log(S(t)) – diagnostyka rozkl. wyklad.')
axes[0].set_xlabel('Czas (dni)')
axes[0].set_ylabel('-log(S(t))')
axes[0].grid(True)

# log(-log(S(t))) vs log(t)
t_idx = kmf.survival_function_.index
valid = (log_s > 0) & (t_idx > 0)
loglog_s = np.log(log_s[valid])
log_t = np.log(t_idx[valid])
axes[1].plot(log_t, loglog_s, 'o-', color='#ff7f0e')
axes[1].set_title('log(-log(S(t))) vs log(t) – diagnostyka Weibulla')
axes[1].set_xlabel('log(Czas)')
axes[1].set_ylabel('log(-log(S(t)))')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('hf_log_survival_diagnostic.png', dpi=150)
plt.show()

# Diagnostyka proporcjonalnosci hazardow (log-log dla grup)
plt.figure(figsize=(10, 6))
for val, label in {0: 'Kobieta', 1: 'Mezczyzna'}.items():
    mask = df['sex'] == val
    kmf_g = KaplanMeierFitter()
    kmf_g.fit(df['time'][mask], df['DEATH_EVENT'][mask], label=label)
    S_g = np.clip(kmf_g.survival_function_.values.flatten(), 1e-10, 1)
    t_g = kmf_g.survival_function_.index
    loglog_g = np.log(-np.log(S_g))
    plt.plot(np.log(t_g), loglog_g, label=label)
plt.title('log(-log(S(t))) wg plci – diagnostyka proporcjonalnosci hazardow')
plt.xlabel('log(Czas)')
plt.ylabel('log(-log(S(t)))')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('hf_loglog_sex.png', dpi=150)
plt.show()


## KOD 4 – Estymator Nelsona-Aalena i wygładzanie kernelowe hazardu

Estymator Nelsona-Aalena: $\hat{H}(t) = \sum_{t_i \leq t} \frac{d_i}{n_i}$. Funkcja hazardu chwilowego jest obliczana przez wygładzanie kernelowe z różnymi pasmami.

In [ ]:
naf = NelsonAalenFitter()
naf.fit(df['time'], df['DEATH_EVENT'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, bw in enumerate([20, 50, 100, 200]):
    ax = axes[i // 2][i % 2]
    hazard_smooth = naf.smoothed_hazard_(bandwidth=bw)
    ax.plot(hazard_smooth, 'r-', linewidth=1.5)
    ax.set_title(f'Wygladzony hazard (bandwidth={bw})')
    ax.set_xlabel('Czas (dni)')
    ax.set_ylabel('Hazard h(t)')
    ax.grid(True)

plt.suptitle('Wygladzanie kernelowe funkcji hazardu – Heart Failure', fontsize=13)
plt.tight_layout()
plt.savefig('hf_hazard_bandwidth.png', dpi=150)
plt.show()


In [ ]:
# Porownanie wygladzonego hazardu miedzy grupami ef_cat
plt.figure(figsize=(12, 7))
colors_ef = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']
cats = df['ef_cat'].cat.categories

for i, cat in enumerate(cats):
    mask = df['ef_cat'] == cat
    if mask.sum() < 10:
        continue
    naf_g = NelsonAalenFitter()
    naf_g.fit(df['time'][mask], df['DEATH_EVENT'][mask])
    hz = naf_g.smoothed_hazard_(bandwidth=50)
    plt.plot(hz, color=colors_ef[i], linewidth=2, label=str(cat))

plt.title('Wygladzony hazard wg frakcji wyrzutowej (EF)\nbandwidth=50')
plt.xlabel('Czas (dni)')
plt.ylabel('Hazard h(t)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('hf_hazard_ef_groups.png', dpi=150)
plt.show()


In [ ]:
# Porownanie hazardu dla zmiennych binarnych na jednym wykresie
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (var, labels_map) in enumerate(binary_vars.items()):
    ax = axes[idx]
    for val in sorted(df[var].unique()):
        mask = df[var] == val
        if mask.sum() < 5:
            continue
        naf_g = NelsonAalenFitter()
        naf_g.fit(df['time'][mask], df['DEATH_EVENT'][mask])
        hz = naf_g.smoothed_hazard_(bandwidth=50)
        ax.plot(hz, linewidth=2,
                label=labels_map.get(val, str(val)))
    ax.set_title(f'Hazard wg {var}')
    ax.set_xlabel('Czas (dni)')
    ax.set_ylabel('Hazard h(t)')
    ax.legend()
    ax.grid(True)

axes[-1].set_visible(False)
plt.suptitle('Wygladzony hazard wg zmiennych binarnych', fontsize=13)
plt.tight_layout()
plt.savefig('hf_hazard_binary_groups.png', dpi=150)
plt.show()
